# M6 · Backfill `fact_notification` (BI superset)

**Goal:** load every notification — not just `SPEED%` — from `staging.notification` into `warehouse.fact_notification` so the BI dashboard can chart geofence, panic, idle, maintenance, route alerts, etc.

The original `fact_speed_notification` (sql/13) stays in place to keep the ML feature contract stable. This is **additive**.

**SQL file:** `sql/17_fact_notification_incr.sql`.
Bound parameters: `:window_start`, `:window_end`, `:etl_run_id`.

**Categorisation:** `description` is bucketed inline into 9 categories so the BI layer can group without parsing strings.

**Exit criteria:** every alert category has rows; `alert_category='other'` < 20% of total.

In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for c in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = c / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings, load_pipeline_config
from accent_fleet.db import get_engine
from sqlalchemy import text

s = settings()
print(f'DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}')

## 2. Inputs

In [ ]:
from datetime import datetime, timedelta
cfg = load_pipeline_config()
MIN_TIME = datetime.fromisoformat(cfg['window']['min_event_time'].replace('Z','+00:00')).replace(tzinfo=None)
CHUNK_DAYS = cfg['window']['backfill_chunk_days']
with get_engine().connect() as conn:
    end_time = conn.execute(text('SELECT MAX(created_at) FROM staging.notification')).scalar_one()
print('staging.notification max created_at =', end_time)

## 3. Execute

In [ ]:
import importlib, time, pandas as pd
import accent_fleet.transforms.facts as facts_module
from accent_fleet.pipeline.run_log import begin_run, end_run
facts_module = importlib.reload(facts_module)
load_fact_incremental = facts_module.load_fact_incremental

run_id = begin_run(mode='notebook:08_load_fact_notification')
progress, cursor, t0 = [], MIN_TIME, time.time()
try:
    while cursor < end_time:
        nxt = min(cursor + timedelta(days=CHUNK_DAYS), end_time)
        res = load_fact_incremental('fact_notification', run_id=run_id, window_end=nxt)
        progress.append({'window_start': cursor, 'window_end': nxt, 'rows_loaded': res.rows_loaded})
        cursor = nxt
    end_run(run_id, status='success', rows_loaded=sum(p['rows_loaded'] for p in progress))
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc)); raise
print(f'done in {time.time()-t0:.1f}s — {len(progress)} chunks')
pd.DataFrame(progress).tail(10)

## 4. Inspect

In [ ]:
import pandas as pd
with get_engine().connect() as conn:
    total = conn.execute(text('SELECT COUNT(*) FROM warehouse.fact_notification')).scalar_one()
    by_cat = pd.read_sql(text('''SELECT alert_category, COUNT(*) AS n
                                  FROM warehouse.fact_notification
                                  GROUP BY 1 ORDER BY 2 DESC'''), conn)
print(f'fact_notification rows: {total:,}')
display(by_cat)
other_pct = by_cat.loc[by_cat['alert_category']=='other','n'].sum() / max(total,1) * 100 if total else 0
print(f"'other' share = {other_pct:.1f}% (target < 20%)")